## Download OC-CCI, and CCMP Data

### Setup and imports

In [7]:
from calendar import monthrange
from datetime import datetime, timezone
from pathlib import Path
import earthaccess
import requests

output_directory = Path("/home/eli2883/ms274/ms274_project/data/")
output_directory.mkdir(parents=True, exist_ok=True)

def monthly_periods():
    """Return the first and last dates of each month, including leap years."""
    for year in range(1997, 2017):
        for month in range(1, 13):
            last_day = monthrange(year, month)[1]
            start = datetime(year, month, 1, tzinfo=timezone.utc)
            end = datetime(year, month, last_day, tzinfo=timezone.utc)
            yield start, end

/home/eli2883/miniconda3/envs/ms274/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### OC-CCI

In [3]:
chlorophyll_directory = output_directory / "chlorophyll"
chlorophyll_directory.mkdir(exist_ok=True)

endpoint = (
    "https://www.oceancolour.org/thredds/ncss/"
    "CCI_ALL-v6.0-1km-DAILY"
)
first_available = datetime(1997, 9, 4, tzinfo=timezone.utc)

for start, end in monthly_periods():
    if end < first_available:
        continue

    start = max(start, first_available)
    destination = chlorophyll_directory / f"occci_scb_{start:%Y%m}.nc"

    if destination.exists():
        continue

    parameters = {
        "var": "chlor_a",
        "north": 35,
        "south": 31.5,
        "west": -121,
        "east": -116.5,
        "time_start": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "time_end": end.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "horizStride": 1,
        "timeStride": 1,
        "accept": "netcdf",
    }

    print(f"Downloading chlorophyll: {start:%Y-%m}")
    temporary_file = destination.with_suffix(".part")

    #handle network errors with as
    with requests.get(
        endpoint, params=parameters, stream=True, timeout=(30, 600)
    ) as response:
        response.raise_for_status()
        with temporary_file.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                output.write(chunk)

    # Only mark the file complete after the download finishes.
    temporary_file.replace(destination)

### CCMP Download

In [8]:
wind_directory = output_directory / "wind"
wind_directory.mkdir(exist_ok=True)
earthaccess.login()

results = earthaccess.search_data(
    short_name="CCMP_WINDS_10M6HR_L4_V3.1",
    temporal=("1997-01-01", "2017-01-01"),
    bounding_box=(-121, 31.5, -116.5, 35),
)

files = earthaccess.download(
    results,
    local_path= wind_directory,
)

QUEUEING TASKS | : 100%|██████████| 7306/7306 [00:00<00:00, 83152.44it/s]
PROCESSING TASKS | : 100%|██████████| 7306/7306 [1:10:53<00:00,  1.72it/s]
COLLECTING RESULTS | : 100%|██████████| 7306/7306 [00:00<00:00, 1308045.63it/s]
